In [2]:
#!/usr/bin/env python3
"""LightGBM random 80/20 holdout for OPERA PV nowcasting.

Reference metrics :
R2=0.9983, MAE=173.47 W, RMSE=360.13 W.

"""
from __future__ import annotations

import argparse
import json
import os
import re
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

SEED = 42
TARGET = "Pa1"
N_JOBS = min(4, os.cpu_count() or 1)
PAPER = {"r2": 0.9983, "mae_w": 173.47, "rmse_w": 360.13}

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path(".").resolve()

DEFAULT_10M = ROOT / "opera_dataset_1w_10min.csv"
DEFAULT_90M = ROOT / "opera_dataset_3w_90min.csv"
DEFAULT_OUT = ROOT / "pv_lightgbm_minimal_results"


CONFIGS = {
    "LGBM_leaf31": dict(
        objective="regression", n_estimators=1000, learning_rate=0.025,
        num_leaves=31, max_depth=-1, min_child_samples=25,
        subsample=0.90, subsample_freq=1, colsample_bytree=0.90,
        reg_lambda=3.0, reg_alpha=0.05,
    ),
    "LGBM_l1_leaf31": dict(
        objective="regression_l1", n_estimators=1000, learning_rate=0.025,
        num_leaves=31, max_depth=-1, min_child_samples=25,
        subsample=0.90, subsample_freq=1, colsample_bytree=0.90,
        reg_lambda=3.0, reg_alpha=0.05,
    ),
}

ENV_PREFIXES = (
    "rad", "irradiance", "globalirradiance",
    "tam", "tamb", "ambienttemp",
    "tmod", "moduletemp", "tempmod", "tempmodule",
)
ENV_SUMMARY_STARTS = ("avg", "mean", "min", "max", "std", "median", "var", "sum", "p10", "p90")
FORBIDDEN_MARKERS = (
    "pa1", "power", "current", "voltage", "output", "generation",
    "production", "energy", "target", "label", "ampere", "watt", "volt",
)


def normalise_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def read_frame(path: Path) -> pd.DataFrame:
    """Read a source CSV; clean timestamps/target and reject duplicate timestamps."""
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    frame = pd.read_csv(path)
    if "timestamp" not in frame.columns or TARGET not in frame.columns:
        raise ValueError(f"{path.name} must contain 'timestamp' and '{TARGET}'.")
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="coerce", utc=True)
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="coerce")
    valid = frame["timestamp"].notna() & np.isfinite(frame[TARGET].to_numpy(dtype=float))
    removed = int((~valid).sum())
    frame = frame.loc[valid].copy().sort_values("timestamp").reset_index(drop=True)
    if removed:
        print(f"{path.name}: removed {removed} row(s) with invalid timestamp/target.")
    if frame["timestamp"].duplicated().any():
        examples = frame.loc[frame["timestamp"].duplicated(keep=False), "timestamp"].head(5).astype(str).tolist()
        raise ValueError(f"Duplicate timestamps in {path.name}: {examples}")
    if len(frame) < 100:
        raise ValueError(f"Too few valid rows in {path.name}: {len(frame)}")
    return frame


def load_and_align(path_10m: Path, path_90m: Path) -> pd.DataFrame:
    """Check target alignment and keep rows with nine consecutive 10-minute timestamps."""
    ten = read_frame(path_10m)
    ninety = read_frame(path_90m)

    overlap = ninety[["timestamp", TARGET]].merge(
        ten[["timestamp", TARGET]], on="timestamp", how="inner", suffixes=("_90m", "_10m")
    )
    if overlap.empty:
        raise ValueError("The 10-minute and 90-minute files have no matching timestamps.")
    diff = np.abs(overlap[f"{TARGET}_90m"].to_numpy(float) - overlap[f"{TARGET}_10m"].to_numpy(float))
    max_diff = float(np.max(diff))
    if max_diff > 1e-3:
        raise ValueError(f"Pa1 differs between files at matching timestamps (max difference {max_diff:g} W).")

    base_ts = pd.DatetimeIndex(ten["timestamp"])
    base_positions = {stamp: i for i, stamp in enumerate(base_ts)}
    step_ns = pd.Timedelta(minutes=10).value
    keep = []
    for i, stamp in enumerate(ninety["timestamp"]):
        pos = base_positions.get(stamp)
        if pos is None or pos < 8:
            continue
        history = base_ts.asi8[pos - 8:pos + 1]
        if len(history) == 9 and np.all(np.diff(history) == step_ns):
            keep.append(i)

    data = ninety.iloc[keep].reset_index(drop=True)
    if data.empty:
        raise ValueError("No rows have complete trailing 10-minute history.")
    print(f"Target-alignment check passed: {len(overlap):,} matching timestamps; max difference={max_diff:g} W.")
    print(f"Rows retained after complete-history check: {len(data):,}")
    return data


def select_environmental_features(data: pd.DataFrame) -> pd.DataFrame:
    """Use environmental columns only; reject output/current/voltage-like names."""
    selected = []
    for col in data.columns:
        if col in {"timestamp", TARGET}:
            continue
        name = normalise_name(col)
        if not name or any(marker in name for marker in FORBIDDEN_MARKERS):
            continue
        is_env = name.startswith(ENV_PREFIXES)
        # Support the paper's irradiance abbreviation GI_avg/GI_min/GI_std.
        is_gi = name == "gi" or (name.startswith("gi") and name[2:].startswith(ENV_SUMMARY_STARTS))
        if is_env or is_gi:
            selected.append(col)

    if not selected:
        raise ValueError(f"No environmental predictors found. Available columns: {list(data.columns)}")
    X = data[selected].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    all_missing = [col for col in X.columns if X[col].isna().all()]
    if all_missing:
        X = X.drop(columns=all_missing)
    if X.shape[1] == 0:
        raise ValueError("No usable environmental predictors remain after numeric conversion.")
    print(f"Environmental predictors ({X.shape[1]}): {list(X.columns)}")
    return X


def build_feature_sets(X_raw: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Keep the original raw and engineered feature sets; no calendar features."""
    sets = {"env_only": X_raw.copy()}
    X = X_raw.copy()
    lookup = {normalise_name(c): c for c in X.columns}

    def find(*aliases: str) -> str | None:
        return next((lookup[normalise_name(a)] for a in aliases if normalise_name(a) in lookup), None)

    rad = find("Rad_avg", "Rad_mean", "GI_avg", "Irradiance_avg")
    tmod = find("Tmod_avg", "Tmodule_avg", "ModuleTemp_avg")
    tamb = find("Tamb_avg", "Tam_avg", "AmbientTemp_avg")
    rad_w2 = find("Rad_avg_w2", "Rad_mean_w2", "GI_avg_w2")
    rad_w3 = find("Rad_avg_w3", "Rad_mean_w3", "GI_avg_w3")

    if rad:
        X["engineered_rad_squared"] = X[rad] ** 2
    if rad and tmod:
        X["engineered_rad_x_module_temp"] = X[rad] * X[tmod]
    if rad and tamb:
        X["engineered_rad_x_ambient_temp"] = X[rad] * X[tamb]
    if tmod and tamb:
        X["engineered_module_minus_ambient_temp"] = X[tmod] - X[tamb]
    if rad and rad_w2:
        X["engineered_rad_delta_w2"] = X[rad] - X[rad_w2]
    if rad and rad_w3:
        X["engineered_rad_delta_w3"] = X[rad] - X[rad_w3]

    sets["env_engineered"] = X.replace([np.inf, -np.inf], np.nan).astype(np.float32)
    print(f"Engineered features added: {len(sets['env_engineered'].columns) - len(X_raw.columns)}")
    return sets


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae_w": float(mean_absolute_error(y_true, y_pred)),
        "rmse_w": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def choose_model(feature_sets: dict[str, pd.DataFrame], y: np.ndarray,
                 fit_idx: np.ndarray, val_idx: np.ndarray) -> tuple[dict, pd.DataFrame]:
    """Select candidate only by validation RMSE, maintaining the original candidate order."""
    rows = []
    candidate_info = {}
    for schema, frame in feature_sets.items():
        for config_name, config in CONFIGS.items():
            name = f"{schema}__{config_name}"
            model = LGBMRegressor(**config, random_state=SEED, n_jobs=N_JOBS, verbosity=-1)
            started = time.perf_counter()
            model.fit(
                frame.iloc[fit_idx], y[fit_idx],
                eval_set=[(frame.iloc[val_idx], y[val_idx])],
                callbacks=[early_stopping(stopping_rounds=60, verbose=False), log_evaluation(0)],
            )
            seconds = time.perf_counter() - started
            best_iter = int(getattr(model, "best_iteration_", 0) or config["n_estimators"])
            pred = model.predict(frame.iloc[val_idx], num_iteration=best_iter)
            score = metrics(y[val_idx], pred)
            rows.append({
                "candidate": name, "schema": schema, "config_name": config_name,
                "best_iteration": best_iter, "validation_fit_seconds": seconds,
                **{f"val_{k}": v for k, v in score.items()},
                "selection_score_validation_rmse_w": score["rmse_w"],
            })
            candidate_info[name] = {
                "candidate": name, "schema": schema, "config_name": config_name,
                "config": config.copy(), "best_iteration": best_iter,
                "feature_columns": list(frame.columns),
            }
            print(f"{name}: validation RMSE={score['rmse_w']:.2f} W; best_iteration={best_iter}")

    table = pd.DataFrame(rows).sort_values("selection_score_validation_rmse_w").reset_index(drop=True)
    winner = candidate_info[str(table.loc[0, "candidate"])].copy()
    winner["validation_metrics"] = {k: float(table.loc[0, f"val_{k}"]) for k in ("r2", "mae_w", "rmse_w")}
    return winner, table


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-10m", type=Path, default=DEFAULT_10M)
    parser.add_argument("--data-90m", type=Path, default=DEFAULT_90M)
    parser.add_argument("--out", type=Path, default=DEFAULT_OUT)
    args = parser.parse_args(args=[])  # Pass empty list to prevent parser.parse_args() from reading Colab sys.argv
    args.out.mkdir(parents=True, exist_ok=True)

    data = load_and_align(args.data_10m, args.data_90m)
    X_raw = select_environmental_features(data)
    feature_sets = build_feature_sets(X_raw)
    y = data[TARGET].to_numpy(dtype=np.float64)


    indices = np.arange(len(y))
    outer_train, test_idx = train_test_split(indices, test_size=0.20, random_state=SEED, shuffle=True)
    fit_idx, val_idx = train_test_split(outer_train, test_size=0.20, random_state=SEED + 1, shuffle=True)
    test_idx = np.sort(np.asarray(test_idx, dtype=int))

    print(f"Rows: {len(data):,}; fit={len(fit_idx):,}; validation={len(val_idx):,}; test={len(test_idx):,}")
    winner, validation_table = choose_model(feature_sets, y, np.asarray(fit_idx), np.asarray(val_idx))
    validation_table.to_csv(args.out / "random_validation_search.csv", index=False)
    print(f"Selected on validation only: {winner['candidate']}")


    final_config = winner["config"].copy()
    final_config["n_estimators"] = winner["best_iteration"]
    selected_frame = feature_sets[winner["schema"]]
    model = LGBMRegressor(**final_config, random_state=SEED, n_jobs=N_JOBS, verbosity=-1)
    started = time.perf_counter()
    model.fit(selected_frame.iloc[outer_train], y[outer_train])
    fit_seconds = time.perf_counter() - started
    pred = model.predict(selected_frame.iloc[test_idx])
    score = metrics(y[test_idx], pred)

    results = pd.DataFrame([{
        "model": "Selected_LightGBM", "selected_candidate": winner["candidate"],
        "fit_seconds": fit_seconds, **score,
        "paper_r2": PAPER["r2"], "paper_mae_w": PAPER["mae_w"], "paper_rmse_w": PAPER["rmse_w"],
        "r2_delta_vs_paper": score["r2"] - PAPER["r2"],
        "mae_reduction_vs_paper_pct": 100 * (PAPER["mae_w"] - score["mae_w"]) / PAPER["mae_w"],
        "rmse_reduction_vs_paper_pct": 100 * (PAPER["rmse_w" ] - score["rmse_w"]) / PAPER["rmse_w"],
        "beats_all_three_paper_metrics": (
            score["r2"] > PAPER["r2"] and score["mae_w"] < PAPER["mae_w"] and score["rmse_w"] < PAPER["rmse_w"]
        ),
        "evaluation_protocol": "random 80/20 holdout; validation split inside outer training set",
        "exact_reproduction_of_paper": False,
    }])
    results.to_csv(args.out / "random_comparison.csv", index=False)

    pd.DataFrame({
        "timestamp": data.iloc[test_idx]["timestamp"].astype(str).to_numpy(),
        "actual_Pa1_W": y[test_idx], "selected_LightGBM_prediction_W": pred,
    }).to_csv(args.out / "random_test_predictions.csv", index=False)

    importance = pd.DataFrame({
        "feature": selected_frame.columns,
        "importance_gain": model.booster_.feature_importance(importance_type="gain"),
        "importance_split": model.booster_.feature_importance(importance_type="split"),
    }).sort_values("importance_gain", ascending=False)
    importance.to_csv(args.out / "random_selected_feature_importance.csv", index=False)

    joblib.dump({
        "model": model, "target": TARGET, "selected_candidate": winner["candidate"],
        "feature_columns": list(selected_frame.columns), "final_config": final_config,
    }, args.out / "selected_lightgbm_model.joblib", compress=0)
    (args.out / "random_recipe.json").write_text(json.dumps({
        "protocol": "random shuffled 80/20 holdout",
        "outer_seed": SEED, "validation_seed": SEED + 1,
        "selected_candidate": winner["candidate"], "selected_config": final_config,
        "validation_metrics": winner["validation_metrics"], "test_metrics": score,
        "n_rows": len(data), "fit_rows": len(fit_idx), "validation_rows": len(val_idx),
        "outer_training_rows": len(outer_train), "test_rows": len(test_idx),
        "selection_objective": "minimum validation RMSE in watts",
        "paper_reference": PAPER,
        "warning": "Paper metrics use 30-fold CV. This random holdout is not a protocol-matched replication; nearby temporal windows may be correlated. If this split was inspected during development, treat it as exploratory.",
    }, indent=2), encoding="utf-8")

    print("\nRANDOM 80/20 TEST RESULTS")
    print(results[["model", "r2", "mae_w", "rmse_w", "beats_all_three_paper_metrics"]].to_string(index=False))
    print(f"Saved outputs to: {args.out.resolve()}")


if __name__ == "__main__":
    main()

Target-alignment check passed: 24,024 matching timestamps; max difference=0 W.
Rows retained after complete-history check: 23,984
Environmental predictors (36): ['Rad_avg', 'Tamb_avg', 'Tmod_avg', 'Rad_max', 'Tamb_max', 'Tmod_max', 'Rad_min', 'Tamb_min', 'Tmod_min', 'Rad_std', 'Tamb_std', 'Tmod_std', 'Rad_avg_w2', 'Tamb_avg_w2', 'Tmod_avg_w2', 'Rad_max_w2', 'Tamb_max_w2', 'Tmod_max_w2', 'Rad_min_w2', 'Tamb_min_w2', 'Tmod_min_w2', 'Rad_std_w2', 'Tamb_std_w2', 'Tmod_std_w2', 'Rad_avg_w3', 'Tamb_avg_w3', 'Tmod_avg_w3', 'Rad_max_w3', 'Tamb_max_w3', 'Tmod_max_w3', 'Rad_min_w3', 'Tamb_min_w3', 'Tmod_min_w3', 'Rad_std_w3', 'Tamb_std_w3', 'Tmod_std_w3']
Engineered features added: 6
Rows: 23,984; fit=15,349; validation=3,838; test=4,797
env_only__LGBM_leaf31: validation RMSE=316.48 W; best_iteration=996
env_only__LGBM_l1_leaf31: validation RMSE=362.28 W; best_iteration=1000
env_engineered__LGBM_leaf31: validation RMSE=311.55 W; best_iteration=1000
env_engineered__LGBM_l1_leaf31: validation RMSE